# SHAP Visualization: Summary, Dependence, and Waterfall Plots

**Learning objectives:**
- Generate SHAP summary plots (beeswarm) from Captum attributions
- Create SHAP dependence plots showing feature interaction effects
- Build waterfall plots showing prediction decomposition for individual inputs
- Produce SHAP force plots for single predictions

**Estimated time:** 15 minutes

**Dataset:** UCI Adult Income (same as notebook 01)

**Visualization types:** Summary/beeswarm, dependence, waterfall, force plot

In [ ]:
learning_objectives(["- Generate SHAP summary plots (beeswarm) from Captum attributions", "Create SHAP dependence plots showing feature interaction effects", "Build waterfall plots showing prediction decomposition for individual inputs", "Produce SHAP force plots for single predictions", "15 minutes", "UCI Adult Income (same as notebook 01)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

from captum.attr import GradientShap

print("Imports successful")

## 1. Reproduce Model and Data from Notebook 01

In [ ]:
# Load data
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
df = adult.frame.copy()
feature_cols = ['age', 'education-num', 'hours-per-week', 'capital-gain',
                'capital-loss', 'fnlwgt', 'workclass', 'marital-status',
                'occupation', 'relationship', 'race', 'sex', 'native-country']
target_col = 'class'
df_processed = df[feature_cols + [target_col]].dropna()
for col in df_processed.select_dtypes(include='category').columns:
    df_processed[col] = LabelEncoder().fit_transform(df_processed[col].astype(str))
X = df_processed[feature_cols].values.astype(np.float32)
y = (df_processed[target_col].astype(str).str.strip() == '>50K').astype(np.float32).values
X_raw = X.copy()  # keep un-scaled for dependence plots
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_raw_test = X_test.copy()
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Define and train model
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 2)
        )
    def forward(self, x): return self.net(x)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

model = TabularMLP(input_dim=X_train.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(30):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(model(X_train_t), y_train_t)
    loss.backward(); opt.step()

model.eval()
print("Model trained. Generating SHAP values for 200 test samples...")

## 2. Compute SHAP Values for a Population of Test Inputs

SHAP summary and dependence plots require attributions across many inputs. We compute GradientSHAP for 200 test samples.

In [ ]:
grad_shap = GradientShap(model)
background = X_train_t[:100]   # 100 background samples

# Compute SHAP values for 200 test samples (in batches for memory)
n_explain = 200
X_explain = X_test_t[:n_explain]
X_explain_raw = X_raw_test[:n_explain]  # original scale for plots

# Batch to avoid memory issues
batch_size = 50
all_attrs = []
for i in range(0, n_explain, batch_size):
    batch = X_explain[i:i+batch_size]
    attrs = grad_shap.attribute(
        inputs=batch,
        baselines=background,
        target=1,
        n_samples=30,
    )
    all_attrs.append(attrs.detach())

shap_values = torch.cat(all_attrs, dim=0).numpy()  # shape: (200, 13)

# Get model predictions
with torch.no_grad():
    probs = torch.softmax(model(X_explain), dim=1)[:, 1].numpy()

print(f"SHAP values computed: {shap_values.shape}")
print(f"Value range: [{shap_values.min():.3f}, {shap_values.max():.3f}]")

## 3. SHAP Summary Plot (Beeswarm)

The beeswarm plot shows the distribution of SHAP values for each feature across all explained inputs. Red = high feature value, blue = low feature value.

In [ ]:
def plot_shap_beeswarm(shap_values, feature_values, feature_names,
                       max_display=13, figsize=(10, 8)):
    """SHAP beeswarm/summary plot from numpy arrays."""
    n_samples, n_features = shap_values.shape

    # Sort features by mean |SHAP|
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    sorted_idx = np.argsort(mean_abs_shap)[::-1][:max_display]

    fig, ax = plt.subplots(figsize=figsize)

    # Normalize feature values to [0, 1] for color mapping
    cmap = plt.cm.RdBu_r

    for plot_pos, feat_idx in enumerate(sorted_idx[::-1]):
        shap_col = shap_values[:, feat_idx]
        feat_col = feature_values[:, feat_idx]

        # Normalize for color
        feat_min, feat_max = feat_col.min(), feat_col.max()
        if feat_max > feat_min:
            feat_norm = (feat_col - feat_min) / (feat_max - feat_min)
        else:
            feat_norm = np.full_like(feat_col, 0.5)

        # Jitter y positions for beeswarm effect
        y_pos = np.full(n_samples, plot_pos)
        np.random.seed(feat_idx)
        jitter_scale = 0.25
        # Bin by SHAP value and jitter within bins
        order = np.argsort(shap_col)
        for j, sample_idx in enumerate(order):
            y_pos[sample_idx] += np.random.uniform(-jitter_scale, jitter_scale)

        colors = cmap(feat_norm)
        ax.scatter(shap_col, y_pos, c=colors, alpha=0.6, s=12, linewidths=0)

    display_names = [feature_names[i] for i in sorted_idx[::-1]]
    ax.set_yticks(range(len(display_names)))
    ax.set_yticklabels(display_names, fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP value (impact on model output)', fontsize=11)
    ax.set_title('SHAP Summary Plot\n(Red=high feature value, Blue=low feature value)',
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
    cbar.set_label('Feature value (normalized)', fontsize=9)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Low', 'Med', 'High'])

    return fig


fig = plot_shap_beeswarm(shap_values, X_explain_raw, feature_cols)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_beeswarm.png")

## 4. SHAP Bar Plot (Global Importance)

The bar plot shows mean |SHAP value| for each feature — the average magnitude of each feature's impact.

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(sorted_idx))[::-1])
bars = ax.barh(
    range(len(sorted_idx)),
    mean_abs_shap[sorted_idx],
    color=colors,
    edgecolor='white'
)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_cols[i] for i in sorted_idx], fontsize=10)
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
ax.set_title('Global Feature Importance (Mean |SHAP|)\nGradientSHAP on 200 test inputs',
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for bar, val in zip(bars, mean_abs_shap[sorted_idx]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_global_importance.png")

## 5. SHAP Dependence Plot

Dependence plots show how the SHAP value for one feature varies with its actual value. Color encodes an interaction feature.

In [ ]:
def plot_shap_dependence(shap_values, feature_values, feature_names,
                          main_feature, interaction_feature, figsize=(8, 5)):
    """SHAP dependence plot with interaction coloring."""
    main_idx = feature_names.index(main_feature)
    int_idx  = feature_names.index(interaction_feature)

    x = feature_values[:, main_idx]
    y = shap_values[:, main_idx]
    color_vals = feature_values[:, int_idx]

    fig, ax = plt.subplots(figsize=figsize)
    scatter = ax.scatter(x, y, c=color_vals, cmap='RdBu_r', alpha=0.6, s=20)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

    # Smooth trend line
    from scipy.ndimage import uniform_filter1d
    sort_order = np.argsort(x)
    x_sorted = x[sort_order]
    y_sorted = y[sort_order]
    window = max(5, len(x) // 20)
    y_smooth = uniform_filter1d(y_sorted, size=window)
    ax.plot(x_sorted, y_smooth, color='black', linewidth=2, label='Trend')

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label(f'{interaction_feature}\n(interaction)', fontsize=9)
    ax.set_xlabel(main_feature, fontsize=11)
    ax.set_ylabel(f'SHAP value for {main_feature}', fontsize=11)
    ax.set_title(f'SHAP Dependence: {main_feature}\nColored by: {interaction_feature}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    return fig


# Plot dependence for top 2 features
top_features_sorted = [feature_cols[i] for i in np.argsort(mean_abs_shap)[::-1]]
top_feat = top_features_sorted[0]
interaction_feat = top_features_sorted[1]  # second most important as interaction

fig = plot_shap_dependence(shap_values, X_explain_raw, feature_cols,
                            main_feature=top_feat,
                            interaction_feature=interaction_feat)
plt.tight_layout()
plt.savefig('shap_dependence_top.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: shap_dependence_top.png")
print(f"Main feature: {top_feat}, Interaction: {interaction_feat}")

## 6. SHAP Dependence Grid: Top 4 Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

top4 = top_features_sorted[:4]

for ax, feat in zip(axes.flatten(), top4):
    feat_idx = feature_cols.index(feat)
    x = X_explain_raw[:, feat_idx]
    y = shap_values[:, feat_idx]

    # Color by feature value itself (self-dependence)
    scatter = ax.scatter(x, y, c=x, cmap='coolwarm', alpha=0.5, s=15)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

    # Add trend
    from scipy.ndimage import uniform_filter1d
    sort_order = np.argsort(x)
    y_smooth = uniform_filter1d(y[sort_order], size=max(5, len(x)//15))
    ax.plot(x[sort_order], y_smooth, 'k-', linewidth=2)

    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel(f'SHAP({feat})', fontsize=10)
    ax.set_title(f'{feat}', fontweight='bold')
    ax.grid(alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Feature value')

plt.suptitle('SHAP Dependence Plots: Top 4 Features\nShowing how each feature\'s attribution varies with its value',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_dependence_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_dependence_grid.png")

## 7. SHAP Waterfall Plot

The waterfall plot shows how each feature contributes to pushing the prediction from the baseline (expected) value to the final prediction for a single input.

In [ ]:
def plot_shap_waterfall(shap_vals_single, feature_values_single, feature_names,
                         baseline, prediction, max_display=10, figsize=(10, 7)):
    """SHAP waterfall plot for a single prediction."""
    n_features = len(shap_vals_single)

    # Sort by absolute value, show top max_display
    abs_vals = np.abs(shap_vals_single)
    sorted_idx = np.argsort(abs_vals)[::-1]

    # Lump remaining features into "other"
    top_idx = sorted_idx[:max_display]
    other_sum = shap_vals_single[sorted_idx[max_display:]].sum()

    top_names = [feature_names[i] for i in top_idx]
    top_vals  = shap_vals_single[top_idx]
    top_fvals = feature_values_single[top_idx]

    if abs(other_sum) > 1e-6 and n_features > max_display:
        top_names = top_names + [f"{n_features - max_display} other features"]
        top_vals  = np.append(top_vals, other_sum)
        top_fvals = np.append(top_fvals, np.nan)

    # Reverse for bottom-up waterfall
    top_names = top_names[::-1]
    top_vals  = top_vals[::-1]
    top_fvals = top_fvals[::-1]

    n_show = len(top_vals)
    running_total = baseline
    starts = []
    for v in top_vals:
        starts.append(running_total)
        running_total += v

    fig, ax = plt.subplots(figsize=figsize)

    colors = ['#d73027' if v > 0 else '#4575b4' for v in top_vals]

    for i, (s, v, c, name, fval) in enumerate(zip(starts, top_vals, colors, top_names, top_fvals)):
        ax.barh(i, v, left=s, color=c, alpha=0.85, height=0.7, edgecolor='white', linewidth=0.5)
        label = f"={fval:.2f}" if not np.isnan(fval) else ""
        sign = '+' if v > 0 else ''
        ax.text(s + v + (0.002 if v > 0 else -0.002), i,
                f"{sign}{v:.4f}",
                ha='left' if v > 0 else 'right',
                va='center', fontsize=8)

    # Connector lines
    for i in range(n_show - 1):
        end_pos = starts[i] + top_vals[i]
        ax.plot([end_pos, end_pos], [i + 0.35, i + 0.65], 'k-', lw=0.7, alpha=0.5)

    ax.set_yticks(range(n_show))
    ax.set_yticklabels(top_names, fontsize=10)
    ax.axvline(baseline, color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'E[f(X)]={baseline:.4f}')
    ax.axvline(prediction, color='black', linestyle='-', linewidth=1.5, alpha=0.8, label=f'f(x)={prediction:.4f}')
    ax.set_xlabel('Model output (P(>50K))', fontsize=11)
    ax.set_title('SHAP Waterfall Plot\nHow features push prediction from baseline to final value',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

    red_patch  = mpatches.Patch(color='#d73027', label='Increases prediction')
    blue_patch = mpatches.Patch(color='#4575b4', label='Decreases prediction')
    ax.legend(handles=[red_patch, blue_patch], fontsize=9, loc='lower right')

    return fig


# Explain the highest-probability input
with torch.no_grad():
    all_probs = torch.softmax(model(X_test_t), dim=1)[:, 1]
    baseline_val = all_probs.mean().item()

high_prob_idx = all_probs.argmax().item()
x_single = X_test_t[[high_prob_idx]]

attrs_single = grad_shap.attribute(
    inputs=x_single,
    baselines=background,
    target=1,
    n_samples=50,
).detach().squeeze().numpy()

pred_val = all_probs[high_prob_idx].item()

fig = plot_shap_waterfall(
    shap_vals_single=attrs_single,
    feature_values_single=X_raw_test[high_prob_idx],
    feature_names=feature_cols,
    baseline=baseline_val,
    prediction=pred_val,
)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: shap_waterfall.png")
print(f"Prediction: {pred_val:.4f}, Baseline: {baseline_val:.4f}, Sum check: {attrs_single.sum():.4f} ≈ {pred_val - baseline_val:.4f}")

## 8. SHAP Force Plot (Horizontal Waterfall)

The force plot is a horizontal version that clearly shows the balance between features pushing toward and away from a prediction.

In [ ]:
def plot_shap_force(shap_vals, feature_values, feature_names, baseline, prediction,
                    max_display=8, figsize=(14, 3)):
    """Horizontal SHAP force plot."""
    # Sort by absolute value
    sorted_idx = np.argsort(np.abs(shap_vals))[::-1][:max_display]
    top_vals  = shap_vals[sorted_idx]
    top_names = [feature_names[i] for i in sorted_idx]
    top_fvals = feature_values[sorted_idx]

    fig, ax = plt.subplots(figsize=figsize)

    pos_vals  = [(n, v, f) for n, v, f in zip(top_names, top_vals, top_fvals) if v > 0]
    neg_vals  = [(n, v, f) for n, v, f in zip(top_names, top_vals, top_fvals) if v < 0]

    y_center = 0
    bar_height = 0.8
    current_x = baseline

    # Draw positive contributions (right side)
    for name, val, fval in sorted(pos_vals, key=lambda x: -x[1]):
        ax.barh(y_center, val, left=current_x, height=bar_height,
                color='#d73027', alpha=0.85, edgecolor='white')
        mid_x = current_x + val / 2
        if abs(val) > 0.01:
            ax.text(mid_x, y_center, f"{name}\n={fval:.1f}",
                    ha='center', va='center', fontsize=7, color='white', fontweight='bold')
        current_x += val

    current_x = baseline
    # Draw negative contributions (left side)
    for name, val, fval in sorted(neg_vals, key=lambda x: x[1]):
        ax.barh(y_center, val, left=current_x, height=bar_height,
                color='#4575b4', alpha=0.85, edgecolor='white')
        mid_x = current_x + val / 2
        if abs(val) > 0.01:
            ax.text(mid_x, y_center, f"{name}\n={fval:.1f}",
                    ha='center', va='center', fontsize=7, color='white', fontweight='bold')
        current_x += val

    ax.axvline(baseline, color='gray', linestyle='--', linewidth=1.5, label=f'Baseline={baseline:.3f}')
    ax.axvline(prediction, color='black', linestyle='-', linewidth=2, label=f'Prediction={prediction:.3f}')
    ax.set_ylim(-1, 1)
    ax.set_yticks([])
    ax.set_xlabel('P(income >50K)', fontsize=11)
    ax.set_title('SHAP Force Plot — Individual Prediction Decomposition', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(axis='x', alpha=0.3)
    return fig


fig = plot_shap_force(
    shap_vals=attrs_single,
    feature_values=X_raw_test[high_prob_idx],
    feature_names=feature_cols,
    baseline=baseline_val,
    prediction=pred_val,
)
plt.tight_layout()
plt.savefig('shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_force_plot.png")

## 9. SHAP Interaction Heatmap: Feature Pairs

Visualize which pairs of features tend to have SHAP values that move together (positive correlation = joint contribution).

In [ ]:
# Correlation matrix of SHAP values (shows which features tend to co-attribute)
shap_corr = np.corrcoef(shap_values.T)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(shap_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
plt.colorbar(im, ax=ax, label='SHAP value correlation')

# Add correlation values
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        val = shap_corr[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color=color)

ax.set_title('SHAP Value Correlation Matrix\n(High correlation = features tend to attribute together)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_interaction_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_interaction_heatmap.png")

## Self-Check Questions

1. **Beeswarm reading:** In the beeswarm plot, what does a cluster of red dots on the positive SHAP side tell you about that feature?

2. **Waterfall efficiency:** Verify the efficiency property: sum the top feature attributions shown in the waterfall plot and add the baseline. Does the result equal the prediction?

3. **Dependence interpretation:** If a dependence plot shows a monotonically increasing trend (higher feature value → higher SHAP value), what does this tell you about the feature's relationship with the target?

4. **Interaction heatmap:** What does a strong negative correlation between two features' SHAP values suggest about how the model treats them?

**Exercise:** Modify the beeswarm plot function to cap the SHAP axis at the 5th and 95th percentile of values, removing outliers. Does this change which features appear most important visually?